In [15]:
# Task 5: read mi_matches.csv -> keep Mumbai Indians wins -> save in SQL table "mi_wins"
# Install once: pip install pandas sqlalchemy pymysql cryptography

from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ---- MySQL settings (only change PASSWORD) ----
USER, PASSWORD, HOST, DB = "root", "Dhara@123", "localhost:3306", "ipl_assignment"

# ---- Step 1: find the CSV automatically (no path needed) ----
home = Path.home()
folders = [Path.cwd(), home / "Desktop", home / "Downloads", home / "Documents", home]
csv_path = None
for folder in folders:
    for name in ("mi_matches.csv", "mi_matches*.csv"):
        found = sorted(folder.glob(name)) if folder.exists() else []
        if found:
            csv_path = found[0]
            break
    if csv_path:
        break
if csv_path is None:
    raise SystemExit("mi_matches.csv not found. Put it in Desktop/Downloads/Documents or beside this script.")
print("Using file:", csv_path)

# ---- Step 2: read CSV and filter Mumbai Indians wins ----
df = pd.read_csv(csv_path)
mi_wins = df[df["winner"] == "Mumbai Indians"]
print("Mumbai Indians wins:", len(mi_wins))

# ---- Step 3: save to SQL table mi_wins ----
try:
    # create database if missing, then connect
    server = create_engine(URL.create("mysql+pymysql", USER, PASSWORD, HOST))
    with server.begin() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB}"))
    server.dispose()
    engine = create_engine(URL.create("mysql+pymysql", USER, PASSWORD, HOST, database=DB))
    print("Saving to MySQL database:", DB)
except Exception as e:
    # fallback so the program never fails (wrong password, MySQL off, etc.)
    print("MySQL not available, using SQLite instead. Reason:", type(e).__name__)
    engine = create_engine("sqlite:///ipl_assignment.db")

mi_wins.to_sql("mi_wins", engine, if_exists="replace", index=False)
print("Data inserted successfully!")

# ---- Step 4: verify ----
result = pd.read_sql(text("SELECT * FROM mi_wins"), engine)
print("Rows in table mi_wins:", len(result))
print(result.head())
engine.dispose()

SystemExit: mi_matches.csv not found. Put it in Desktop/Downloads/Documents or beside this script.